#### **1. Import Library**

In [1]:
import json
import sys
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

# Menentukan folder utama project.
candidate_roots = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in candidate_roots
        if (path / "scripts" / "feature_engineering.py").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder scripts tidak ditemukan. "
        "Pastikan notebook berada di dalam project."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.feature_engineering import engineer_features

print("Project root:", PROJECT_ROOT)
print("Import feature engineering berhasil.")

Project root: D:\SAINS DATA ITS\SEMESTER 5\Supervised Machine Learning\structural-damage-classification
Import feature engineering berhasil.


#### **2. Load Dataset, Model, dan Metadata**

In [2]:
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODEL_DIR / "final_damage_model.joblib"
METADATA_PATH = MODEL_DIR / "model_metadata.json"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"
TEST_PATH = DATA_DIR / "test.csv"
TRAIN_PATH = DATA_DIR / "train.csv"

required_files = [
    MODEL_PATH,
    METADATA_PATH,
    SAMPLE_PATH,
    TEST_PATH,
    TRAIN_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {path}"
        )

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)
sample_submission = pd.read_csv(SAMPLE_PATH)

final_model = joblib.load(MODEL_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

TARGET_COLUMN = metadata["target_column"]
FEATURE_COLUMNS = metadata["feature_columns"]
ID_COLUMNS = metadata.get("id_columns", [])

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Target:", TARGET_COLUMN)
print("Model:", metadata["best_model_name"])

display(test_df.head())

Train: (533465, 17)
Test: (228629, 16)
Target: damage_grade
Model: LightGBM


,building_id,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,superstructure
0,122207000141,12,1205,120504,1,8,240,10,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,has_superstructure_mud_mortar_stone
1,284909000871,28,2810,281006,3,4,384,22,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
2,241904000231,24,2409,240909,2,999,384,21,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
3,248308000891,24,2412,241202,2,16,527,19,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
4,300405002981,30,3001,300104,2,6,280,16,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Timber-Planck,Not attached,Rectangular,has_superstructure_mud_mortar_stone


#### **3. Data Preparation**

In [3]:
# Memastikan kolom target tersedia pada template.
if TARGET_COLUMN not in sample_submission.columns:
    raise ValueError(
        f"Kolom target '{TARGET_COLUMN}' tidak ditemukan "
        "pada sample_submission.csv."
    )

# Memastikan jumlah baris sesuai.
if len(test_df) != len(sample_submission):
    raise ValueError(
        f"Jumlah baris test ({len(test_df)}) tidak sama "
        f"dengan template ({len(sample_submission)})."
    )

# Menentukan kolom ID untuk menyelaraskan urutan data.
shared_id_columns = [
    col for col in ID_COLUMNS
    if col in test_df.columns
    and col in sample_submission.columns
    and col != TARGET_COLUMN
]

# Jika metadata tidak menunjukkan ID, gunakan kolom non-target
# dari template yang juga tersedia di test dan memiliki nilai unik.
if not shared_id_columns:
    shared_id_columns = [
        col for col in sample_submission.columns
        if col != TARGET_COLUMN
        and col in test_df.columns
        and test_df[col].is_unique
        and sample_submission[col].is_unique
    ]

if shared_id_columns:
    if test_df[shared_id_columns].isna().any().any():
        raise ValueError(
            "Kolom ID pada test memiliki missing values."
        )

    if sample_submission[shared_id_columns].isna().any().any():
        raise ValueError(
            "Kolom ID pada template memiliki missing values."
        )

    if test_df.duplicated(subset=shared_id_columns).any():
        raise ValueError("ID pada test tidak unik.")

    if sample_submission.duplicated(
        subset=shared_id_columns
    ).any():
        raise ValueError("ID pada template tidak unik.")

    ordered_test = sample_submission[
        shared_id_columns
    ].merge(
        test_df,
        on=shared_id_columns,
        how="left",
        sort=False,
        validate="one_to_one",
        indicator=True
    )

    if (ordered_test["_merge"] != "both").any():
        raise ValueError(
            "Ada ID pada template yang tidak ditemukan di test."
        )

    test_df = ordered_test.drop(columns="_merge")

    print("Urutan data test disesuaikan dengan template.")
else:
    print(
        "Kolom ID bersama tidak ditemukan. "
        "Urutan baris test asli dipertahankan."
    )

# Memastikan seluruh fitur training tersedia.
missing_features = [
    col for col in FEATURE_COLUMNS
    if col not in test_df.columns
]

if missing_features:
    raise ValueError(
        f"Fitur test tidak ditemukan: {missing_features}"
    )

X_submission = engineer_features(
    test_df[FEATURE_COLUMNS],
    expected_columns=FEATURE_COLUMNS
)

print("Jumlah data untuk prediksi:", len(X_submission))
print("Jumlah fitur:", len(FEATURE_COLUMNS))

display(X_submission.head())

Urutan data test disesuaikan dengan template.
Jumlah data untuk prediksi: 228629
Jumlah fitur: 15


,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,superstructure
0,12,1205,120504,1,8,240,10,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Not attached,Rectangular,has_superstructure_mud_mortar_stone
1,28,2810,281006,3,4,384,22,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
2,24,2409,240909,2,999,384,21,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
3,24,2412,241202,2,16,527,19,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,has_superstructure_mud_mortar_stone
4,30,3001,300104,2,6,280,16,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Timber-Planck,Not attached,Rectangular,has_superstructure_mud_mortar_stone


#### **4. Prediksi Data Test**

In [4]:
predictions = final_model.predict(X_submission)

if len(predictions) != len(test_df):
    raise ValueError(
        "Jumlah prediksi tidak sesuai dengan jumlah data test."
    )

if pd.isna(predictions).any():
    raise ValueError(
        "Prediksi mengandung missing values."
    )

# Memastikan label prediksi dikenal dari data training.
known_labels = set(
    train_df[TARGET_COLUMN].dropna().unique()
)

predicted_labels = set(pd.Series(predictions).unique())

unknown_labels = predicted_labels - known_labels

if unknown_labels:
    raise ValueError(
        f"Ditemukan label prediksi yang tidak dikenal: "
        f"{unknown_labels}"
    )

print("Prediksi selesai.")
print("Jumlah prediksi:", len(predictions))

print("\nDistribusi hasil prediksi:")
display(
    pd.Series(
        predictions,
        name=TARGET_COLUMN
    ).value_counts().sort_index().to_frame("Jumlah")
)

Prediksi selesai.
Jumlah prediksi: 228629

Distribusi hasil prediksi:


,Jumlah
damage_grade,
Grade 1,32621
Grade 2,31881
Grade 3,39243
Grade 4,53800
Grade 5,71084


#### **5. Membuat File Submission**

In [5]:
#### Membuat File Submission

# Salin template submission resmi.
submission = sample_submission.copy()

# Mengubah label "Grade 1" menjadi 1, "Grade 2" menjadi 2, dst.
predictions_numeric = pd.Series(predictions).astype(str).str.extract(
    r"(\d+)",
    expand=False
)

# Pastikan semua prediksi berhasil dikonversi.
if predictions_numeric.isna().any():
    invalid_labels = pd.Series(predictions)[
        predictions_numeric.isna().to_numpy()
    ].unique()

    raise ValueError(
        f"Label tidak dapat dikonversi menjadi angka: {invalid_labels}"
    )

predictions_numeric = predictions_numeric.astype(int)

# Masukkan label numerik ke template.
submission[TARGET_COLUMN] = predictions_numeric.to_numpy()

# Pertahankan urutan kolom sesuai template Kaggle.
submission = submission[sample_submission.columns]

# Validasi akhir.
if len(submission) != len(test_df):
    raise ValueError("Jumlah baris submission tidak sesuai.")

if submission.isna().any().any():
    raise ValueError("Submission mengandung missing values.")

if not pd.api.types.is_numeric_dtype(submission[TARGET_COLUMN]):
    raise ValueError("Kolom target masih bukan numerik.")

# Simpan submission.
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = SUBMISSION_DIR / "submission.csv"

submission.to_csv(SUBMISSION_PATH, index=False)

print("Submission berhasil dibuat.")
print("Lokasi:", SUBMISSION_PATH)
print("\nContoh hasil submission:")
display(submission.head(10))

print("\nNilai target:", sorted(submission[TARGET_COLUMN].unique()))
print("Tipe target:", submission[TARGET_COLUMN].dtype)

Submission berhasil dibuat.
Lokasi: D:\SAINS DATA ITS\SEMESTER 5\Supervised Machine Learning\structural-damage-classification\submissions\submission.csv

Contoh hasil submission:


,building_id,damage_grade
0,122207000141,2
1,284909000871,4
2,241904000231,3
3,248308000891,3
4,300405002981,4
5,313209000421,1
6,365001030461,3
7,300107000341,4
8,224001000142,5
9,222708000711,5



Nilai target: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Tipe target: int64


#### **6. Pemeriksaan File Submission**

In [6]:
submission_check = pd.read_csv(SUBMISSION_PATH)

assert len(submission_check) == len(test_df), (
    "Jumlah baris submission tidak sesuai."
)

assert submission_check.columns.tolist() == (
    sample_submission.columns.tolist()
), "Nama atau urutan kolom tidak sesuai template."

assert not submission_check.isna().any().any(), (
    "Submission mengandung missing values."
)

print("Pemeriksaan berhasil.")
print("File siap diperiksa dan diunggah ke Kaggle.")
print(f"Lokasi: {SUBMISSION_PATH}")

display(submission_check.head())

Pemeriksaan berhasil.
File siap diperiksa dan diunggah ke Kaggle.
Lokasi: D:\SAINS DATA ITS\SEMESTER 5\Supervised Machine Learning\structural-damage-classification\submissions\submission.csv


,building_id,damage_grade
0,122207000141,2
1,284909000871,4
2,241904000231,3
3,248308000891,3
4,300405002981,4
